<a href="https://colab.research.google.com/github/Kate6097/train/blob/important-functions/analyze_and_set_layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def analyze_and_set_layers(generator, image, text, clip_loss, num_trainable=15):
    """
    АНАЛИЗИРУЕТ, какие модули (слои) самые важные, и устанавливает им
    requires_grad=True, а всем остальным — False.

    Аргументы:
    generator: ваш StyleGAN2 Generator.
    image: сгенерированное изображение.
    text: текстовое описание для CLIP (выход)
    clip_loss: функция для вычисления CLIP-лосса.
    num_trainable: количество модулей, которые нужно разморозить.
    """

    for param in generator.parameters():
        param.requires_grad_(True)
        param.grad = None

    #Вычисляем CLIP-лосс и градиенты по нему
    clip_image = normalize(resize(image))
    text_tensor = clip.tokenize([text]).to(image.device)
    clip_value = clip_loss(clip_image, text_tensor)
    clip_value.backward()

    #Собираем и агрегируем важность по логическим модулям
    module_importance = collections.defaultdict(float)
    module_param_count = collections.defaultdict(int)
    debug_module_contents = collections.defaultdict(list)

    target_module_types = (
        type(generator.conv1),
        type(generator.to_rgb1),
        type(generator.style[0]),
        type(generator.style[1]),
        type(generator.input)
    )

    # Проходимся по всем именованным модулям
    for module_name, module in generator.named_modules():
        if module_name == '':
            continue
        is_target_module = False
        if isinstance(module, target_module_types):
            is_target_module = True
        current_module_grad_sum = 0.0
        current_module_param_count = 0

        for param_name, param in module.named_parameters(recurse=False):
            if param.grad is not None:
                current_module_grad_sum += param.grad.abs().mean().item()
                current_module_param_count += 1
                debug_module_contents[module_name].append(f"{module_name}.{param_name}")

        if current_module_param_count > 0:
            module_importance[module_name] = current_module_grad_sum / current_module_param_count
            module_param_count[module_name] = current_module_param_count


    module_importance_new = collections.defaultdict(float)
    module_param_count_new = collections.defaultdict(int)
    debug_module_contents_new = collections.defaultdict(list)
    all_named_modules = dict(generator.named_modules())

    logical_layer_classes = []
    if hasattr(generator, 'conv1') and isinstance(generator.conv1, nn.Module):
        logical_layer_classes.append(type(generator.conv1)) # StyledConv
    if hasattr(generator, 'to_rgb1') and isinstance(generator.to_rgb1, nn.Module):
        logical_layer_classes.append(type(generator.to_rgb1)) # ToRGB
    if hasattr(generator.style, '0') and isinstance(generator.style[0], nn.Module):
        # PixelNorm or EqualLinear
        logical_layer_classes.append(type(generator.style[0]))
    if hasattr(generator, 'input') and isinstance(generator.input, nn.Module):
        logical_layer_classes.append(type(generator.input)) # ConstantInput

    # Дополнительно: EqualLinear из style MLP
    for i in range(len(generator.style)):
        if isinstance(generator.style[i], nn.Module) and type(generator.style[i]) not in logical_layer_classes:
            logical_layer_classes.append(type(generator.style[i])) # EqualLinear


    for name, param in generator.named_parameters():
        if param.grad is not None:
            parts = name.split('.')
            found_logical_module_name = None

            for i in range(len(parts)):
                sub_module_name = '.'.join(parts[:i+1])
                if sub_module_name in all_named_modules:
                    current_sub_module_instance = all_named_modules[sub_module_name]
                    if isinstance(current_sub_module_instance, tuple(logical_layer_classes)):
                        found_logical_module_name = sub_module_name
                        break
            if found_logical_module_name is None:

                if '.' in name:
                    found_logical_module_name = '.'.join(parts[:-1])
                else:
                    found_logical_module_name = name


            module_importance_new[found_logical_module_name] += param.grad.abs().mean().item()
            module_param_count_new[found_logical_module_name] += 1
            debug_module_contents_new[found_logical_module_name].append(name) # Для отладки

    # Вычисляем среднюю важность для каждого модуля (уже сделано в цикле для module_importance_new)
    for name in module_importance_new:
        module_importance_new[name] /= module_param_count_new[name]

    sorted_modules = sorted(module_importance_new.items(), key=lambda x: x[1], reverse=True)

    trainable_modules = {x[0] for x in sorted_modules[:num_trainable]}


    #print("\n--- НОВАЯ РАЗБИВКА ПО ЛОГИЧЕСКИМ МОДУЛЯМ (ИСПРАВЛЕНО) ---")
    #print("Разбивка по логическим модулям (имена модулей -> количество параметров):")
    #single_param_modules = []
    #for mod_name, param_list in debug_module_contents_new.items():
    #    if len(param_list) == 1:
    #        single_param_modules.append((mod_name, param_list[0]))
    #    print(f"  Модуль: '{mod_name}' -> Параметров: {len(param_list)}")

    #if single_param_modules:
    #    print("\nВНИМАНИЕ (НОВЫЙ ПОДХОД): Следующие 'модули' содержат только 1 параметр (это может быть нормально для PixelNorm/ConstantInput):")
    #    for mod_name, param_name in single_param_modules:
    #        print(f"  - Модуль '{mod_name}' содержит только параметр '{param_name}'")

    #print(f"\nВсего уникальных логических модулей, определенных для градиентов: {len(module_importance_new)}")
    #

    #замораживаем всё, а потом размораживаем только выбранные модули
    trainable_params_count = 0
    for name, param in generator.named_parameters():
        parts = name.split('.')
        current_param_logical_module = None
        for i in range(len(parts)):
            sub_module_name = '.'.join(parts[:i+1])
            if sub_module_name in all_named_modules:
                current_sub_module_instance = all_named_modules[sub_module_name]
                if isinstance(current_sub_module_instance, tuple(logical_layer_classes)):
                    current_param_logical_module = sub_module_name
                    break

        if current_param_logical_module is None:
            if '.' in name:
                current_param_logical_module = '.'.join(parts[:-1])
            else:
                current_param_logical_module = name


        is_trainable = current_param_logical_module in trainable_modules
        param.requires_grad_(is_trainable)
        if is_trainable:
            trainable_params_count += 1

    print(f"\nАнализ завершен. Выбрано для обучения {len(trainable_modules)} логических модулей (всего параметров: {trainable_params_count}).")

    return clip_value.item(), list(trainable_modules)

